# SQL + RAG + Chatbot Test Notebook

Use this notebook to verify that SQL Server is connected, RAG sources are ingested, and the chatbot can call tools.

## 1. Load Settings

In [1]:
from config.settings import Settings

settings = Settings()
{
    'llm_provider': settings.llm_provider,
    'llm_model': settings.llm_model,
    'sqlserver_host': settings.sqlserver_host,
    'sqlserver_database': settings.sqlserver_database,
    'sqlserver_driver': settings.sqlserver_driver,
    'trusted_connection': settings.sqlserver_trusted_connection,
    'encrypt': settings.sqlserver_encrypt,
    'qdrant_enabled': settings.qdrant_enabled,
    'qdrant_url': settings.qdrant_url,
}

{'llm_provider': 'mock',
 'llm_model': 'gpt-4.1',
 'sqlserver_host': None,
 'sqlserver_database': 'Subscription_Analysis',
 'sqlserver_driver': 'ODBC Driver 18 for SQL Server',
 'trusted_connection': True,
 'encrypt': False,
 'qdrant_enabled': False,
 'qdrant_url': 'http://localhost:6333'}

## 2. Check Installed ODBC Drivers

In [2]:
import pyodbc

pyodbc.drivers()

['SQL Server',
 'SQL Server Native Client RDA 11.0',
 'ODBC Driver 17 for SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)',
 'Microsoft Access Text Driver (*.txt, *.csv)']

If you do not see `ODBC Driver 18 for SQL Server`, either install Microsoft ODBC Driver 18 or change `SQLSERVER_DRIVER` in `.env` to an installed driver, for example `ODBC Driver 17 for SQL Server`.

## 3. SQL Tool Diagnostics

In [3]:
from tools.sql_tool import SqlServerTool

sql_tool = SqlServerTool(settings)
sql_tool.diagnostics()

{'configured': False,
 'dsn': None,
 'host': None,
 'database': 'Subscription_Analysis',
 'driver': 'ODBC Driver 18 for SQL Server',
 'trusted_connection': True,
 'encrypt': False,
 'connected': False,
 'installed_drivers': ['SQL Server',
  'SQL Server Native Client RDA 11.0',
  'ODBC Driver 17 for SQL Server',
  'Microsoft Access Driver (*.mdb, *.accdb)',
  'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
  'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)',
  'Microsoft Access Text Driver (*.txt, *.csv)'],
 'connection_string_preview': 'DRIVER={ODBC Driver 18 for SQL Server};SERVER=None;DATABASE=Subscription_Analysis;Trusted_Connection=yes;Encrypt=no;TrustServerCertificate=yes;',
 'error': 'SQL Server is not configured. Set SQLSERVER_HOST and SQLSERVER_DATABASE in .env, then restart the API.'}

## 4. Direct SQL Schema Query

In [4]:
schema_sql = """
SELECT TABLE_NAME, COLUMN_NAME
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME IN ('Society_Master_Data','Subscription_Plans','Subscription_Transactions','Payment_Gateway','Complaint_tickets')
ORDER BY TABLE_NAME, ORDINAL_POSITION
"""

sql_tool.validate_sql(schema_sql)
rows = sql_tool.execute_select(schema_sql)
rows[:10], len(rows)

([{'notice': 'SQL Server is not configured. Set SQLSERVER_DSN or host credentials.',
   'sql': "\nSELECT TABLE_NAME, COLUMN_NAME\nFROM INFORMATION_SCHEMA.COLUMNS\nWHERE TABLE_NAME IN ('Society_Master_Data','Subscription_Plans','Subscription_Transactions','Payment_Gateway','Complaint_tickets')\nORDER BY TABLE_NAME, ORDINAL_POSITION\n"}],
 1)

## 5. Direct Business SQL Query

In [ ]:
business_sql = sql_tool.generate_sql('Which subscription plans have the highest active contract value?')
business_sql, sql_tool.execute_select(business_sql)

## 6. Local RAG Ingestion and Retrieval

In [5]:
from pathlib import Path
from rag.pipeline import RagPipeline

rag = RagPipeline(settings)
sources = [Path('sample_data/subscription_analysis_knowledge.md'), Path('output/pdf/subscription_analysis_demo.pdf')]
ingested = {str(path): rag.ingest_path(path) for path in sources if path.exists()}
hits = rag.retrieve('What is payment leakage and SLA breach?', top_k=5)
ingested, [(chunk.id, score, chunk.text[:300]) for chunk, score in hits]

({}, [])

## 7. End-to-End Orchestrator Test Without Streamlit

In [6]:
from app.dependencies import get_orchestrator
from models.schemas import ChatRequest

orchestrator = get_orchestrator()
orchestrator.rag.ingest_path(Path('sample_data/subscription_analysis_knowledge.md'))
if Path('output/pdf/subscription_analysis_demo.pdf').exists():
    orchestrator.rag.ingest_path(Path('output/pdf/subscription_analysis_demo.pdf'))

response = await orchestrator.chat(ChatRequest(
    query='Retrieve the SLA breach definition from RAG and show complaint tickets that breached SLA from SQL',
    session_id='notebook-e2e',
    files=['output/pdf/subscription_analysis_demo.pdf'],
))
response.model_dump()

FileNotFoundError: [Errno 2] No such file or directory: 'sample_data\\subscription_analysis_knowledge.md'

## 8. API Test Against Running FastAPI

In [ ]:
import httpx

base_url = 'http://127.0.0.1:8000'
print(httpx.get(f'{base_url}/config/status', timeout=10).json())
print(httpx.get(f'{base_url}/sql/health', timeout=20).json())

payload = {
    'query': 'Which payments failed or are pending from SQL?',
    'session_id': 'api-sql-test',
    'files': []
}
httpx.post(f'{base_url}/chat', json=payload, timeout=120).json()